# TF-IDF


## Library

In [3]:
!pip install plotly
!pip install --upgrade gensim


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

# ======================
# STEP 2: Load Data CSV
# ======================
# kata dan jumlah
df_kata = pd.read_csv("kata_jumlah.csv")

# abstrak yang sudah clean
df_abstrak = pd.read_csv("pta_word_frequency_clean.csv")

print("kata_jumlah.csv:")
print(df_kata.head(), "\n")

print("pta_word_frequency_clean.csv:")
print(df_abstrak.head())

kata_jumlah.csv:
         kata  jumlah
0      sistem    1568
1      metode    1397
2       hasil    1167
3        data    1081
4  penelitian     866 

pta_word_frequency_clean.csv:
                                       abstrak_clean  \
0  sistem informasi akademik siakad merupakan sis...   
1  berjalannya koneksi jaringan komputer dengan l...   
2  web server adalah sebuah perangkat lunak serve...   
3  penjadwalan kuliah di perguruan tinggi merupak...   
4  seiring perkembangan teknologi yang ada diduni...   

                                               clean  
0  sistem informasi akademik siakad sistem inform...  
1  berjalannya koneksi jaringan komputer lancar g...  
2  web server perangkat lunak server berfungsi me...  
3  penjadwalan kuliah perguruan kompleks permasal...  
4  seiring perkembangan teknologi didunia muncul ...  


In [5]:
df_kata['kata'] = df_kata['kata'].astype(str)
df_abstrak['clean'] = df_abstrak['clean'].astype(str)

In [6]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_abstrak['clean'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("TF-IDF Matrix:")
print(tfidf_df.head())

# Simpan ke CSV
tfidf_df.to_csv("hasil_tfidf.csv", index=False)


TF-IDF Matrix:
   aalysis  aam   ab  abad  abadi  ability  abjad  absensi  absolut  absolute  \
0      0.0  0.0  0.0   0.0    0.0      0.0    0.0      0.0      0.0       0.0   
1      0.0  0.0  0.0   0.0    0.0      0.0    0.0      0.0      0.0       0.0   
2      0.0  0.0  0.0   0.0    0.0      0.0    0.0      0.0      0.0       0.0   
3      0.0  0.0  0.0   0.0    0.0      0.0    0.0      0.0      0.0       0.0   
4      0.0  0.0  0.0   0.0    0.0      0.0    0.0      0.0      0.0       0.0   

   ...  zara  zat  zcz   zf  zona  zone  zoning  zoom  zucara  zungu  
0  ...   0.0  0.0  0.0  0.0   0.0   0.0     0.0   0.0     0.0    0.0  
1  ...   0.0  0.0  0.0  0.0   0.0   0.0     0.0   0.0     0.0    0.0  
2  ...   0.0  0.0  0.0  0.0   0.0   0.0     0.0   0.0     0.0    0.0  
3  ...   0.0  0.0  0.0  0.0   0.0   0.0     0.0   0.0     0.0    0.0  
4  ...   0.0  0.0  0.0  0.0   0.0   0.0     0.0   0.0     0.0    0.0  

[5 rows x 8796 columns]


In [7]:
corpus = []
for col in df_abstrak['clean']:
    word_list = col.split(" ")
    corpus.append(word_list)

# train model word2vec
model = Word2Vec(corpus, min_count=1, vector_size=56, workers=4)

# contoh: cari kata mirip
if 'penelitian' in model.wv:
    print("Kata mirip dengan 'penelitian':")
    print(model.wv.most_similar('penelitian', topn=5))

# Simpan vector word2vec ke CSV
word_vectors = pd.DataFrame(
    [model.wv[word] for word in model.wv.index_to_key],
    index=model.wv.index_to_key
)
word_vectors.to_csv("hasil_word2vec.csv")

Kata mirip dengan 'penelitian':
[('pengolahan', 0.9993386268615723), ('dibandingkan', 0.9992337226867676), ('bobot', 0.9992331266403198), ('optimal', 0.999090313911438), ('database', 0.9990566968917847)]


In [8]:
# ======================
# STEP 5b: Mean Embedding Vectorizer
# ======================

class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

# Gunakan untuk abstrak
vec = MeanEmbeddingVectorizer(model)
abstrak_embeddings = vec.fit_transform(df_abstrak['clean'])

print("Shape embeddings:", abstrak_embeddings.shape)


Shape embeddings: (858, 56)


In [9]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df_abstrak['clean'])

In [10]:
df_abstrak['array'] = list(mean_embedded)

print(df_abstrak.head(5))

                                       abstrak_clean  \
0  sistem informasi akademik siakad merupakan sis...   
1  berjalannya koneksi jaringan komputer dengan l...   
2  web server adalah sebuah perangkat lunak serve...   
3  penjadwalan kuliah di perguruan tinggi merupak...   
4  seiring perkembangan teknologi yang ada diduni...   

                                               clean  \
0  sistem informasi akademik siakad sistem inform...   
1  berjalannya koneksi jaringan komputer lancar g...   
2  web server perangkat lunak server berfungsi me...   
3  penjadwalan kuliah perguruan kompleks permasal...   
4  seiring perkembangan teknologi didunia muncul ...   

                                               array  
0  [-0.30112004, 0.34271467, 0.047104325, 0.08555...  
1  [-0.2787827, 0.2866456, 0.017059306, 0.130994,...  
2  [-0.25317279, 0.2764192, 0.013476692, 0.129002...  
3  [-0.35891527, 0.40646052, 0.059396055, 0.09264...  
4  [-0.2535674, 0.26926145, 0.014926432, 0.119040..

In [11]:
# Cek panjang embedding
df_abstrak['embedding_length'] = df_abstrak['array'].str.len()
print(df_abstrak['embedding_length'])
print("Shape abstrak + embedding:", df_abstrak.shape)

0      56
1      56
2      56
3      56
4      56
       ..
853    56
854    56
855    56
856    56
857    56
Name: embedding_length, Length: 858, dtype: int64
Shape abstrak + embedding: (858, 4)


In [12]:
# Ambil jumlah fitur dari embedding
num_features = len(df_abstrak['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris embedding
for embedding_list in df_abstrak['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df.head())
print("Shape embedding_df:", embedding_df.shape)

         f1        f2        f3        f4        f5        f6        f7  \
0 -0.301120  0.342715  0.047104  0.085554  0.004305 -0.737247 -0.008002   
1 -0.278783  0.286646  0.017059  0.130994 -0.028938 -0.705906 -0.011270   
2 -0.253173  0.276419  0.013477  0.129002 -0.018096 -0.656576 -0.007254   
3 -0.358915  0.406461  0.059396  0.092644  0.042917 -0.869529  0.007809   
4 -0.253567  0.269261  0.014926  0.119040 -0.043107 -0.668507  0.000301   

         f8        f9       f10  ...       f47       f48       f49       f50  \
0 -0.937851 -0.673244 -0.508182  ...  0.455717  0.290244  0.194228  0.216686   
1 -0.907701 -0.664114 -0.472518  ...  0.431918  0.256486  0.208003  0.200306   
2 -0.865269 -0.613529 -0.449692  ...  0.422610  0.245858  0.191778  0.190394   
3 -1.113279 -0.760118 -0.570090  ...  0.545187  0.354879  0.223079  0.236863   
4 -0.830214 -0.635567 -0.447145  ...  0.403054  0.229033  0.179971  0.178604   

        f51       f52       f53       f54       f55       f56  
0 -0